In [ ]:
%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = None
%sql sqlite:///../data/data.db
%config SqlMagic.feedback = 0

In [ ]:
# %sql DROP TABLE IF EXISTS search_results;

In [ ]:
%%sql 
CREATE TABLE IF NOT EXISTS videos (
    kind TEXT,
    etag TEXT,
    id TEXT,
    snippet_publishedAt TEXT,
    snippet_channelId TEXT,
    snippet_title TEXT,
    snippet_description TEXT,
    snippet_thumbnails_default_url TEXT,
    snippet_thumbnails_default_width TEXT,
    snippet_thumbnails_default_height TEXT,
    snippet_thumbnails_medium_url TEXT,
    snippet_thumbnails_medium_width TEXT,
    snippet_thumbnails_medium_height TEXT,
    snippet_thumbnails_high_url TEXT,
    snippet_thumbnails_high_width TEXT,
    snippet_thumbnails_high_height TEXT,
    snippet_thumbnails_standard_url TEXT,
    snippet_thumbnails_standard_width TEXT,
    snippet_thumbnails_standard_height TEXT,
    snippet_thumbnails_maxres_url TEXT,
    snippet_thumbnails_maxres_width TEXT,
    snippet_thumbnails_maxres_height TEXT,
    snippet_channelTitle TEXT,
    snippet_tags TEXT,
    snippet_categoryId TEXT,
    snippet_liveBroadcastContent TEXT,
    snippet_localized_title TEXT,
    snippet_localized_description TEXT,
    snippet_defaultAudioLanguage TEXT,
    contentDetails_duration TEXT,
    contentDetails_dimension TEXT,
    contentDetails_definition TEXT,
    contentDetails_caption TEXT,
    contentDetails_licensedContent TEXT,
    contentDetails_projection TEXT,
    statistics_viewCount TEXT,
    statistics_likeCount TEXT,
    statistics_favoriteCount TEXT,
    statistics_commentCount TEXT,
    topicDetails_topicCategories TEXT,
    contentDetails_contentRating_ytRating TEXT,
    contentDetails_regionRestriction_blocked TEXT,
    timestamp TEXT,
    query TEXT
);

In [ ]:
%%sql 
CREATE TABLE IF NOT EXISTS channel_ids (
    channel_id TEXT,
    written TEXT
);

In [ ]:
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build

from tqdm.notebook import tqdm as tqdm

import pandas as pd

import sys

from sqlalchemy import create_engine

def get_channel_ids(query_string, api_keys):
    for api_key in api_keys:
        try:
            youtube = build('youtube', 'v3', developerKey=api_key)
            response = youtube.search().list(
                maxResults=50,
                part='snippet',
                q=query_string,
                relevanceLanguage='en'
            ).execute()
            return [item['snippet']['channelId'] for item in response['items']]
        except HttpError as e:
            if e.resp.status == 403:
                continue
            else:
                raise e
    raise Exception("All API keys have exceeded their quota.")

def get_all_channel_ids(query_string, api_keys, verbose=False):
    channel_ids = []
    next_page_token = None
    current_key_index = 0
    ix = 0
    while True:
        sys.stdout.write(f"\r{ix}")
        sys.stdout.flush()
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])
            
            response = youtube.search().list(
                maxResults=50,
                part='snippet',
                q=query_string,
                relevanceLanguage='en',
                pageToken=next_page_token
            ).execute()
            
            channel_ids.extend([item['snippet']['channelId'] for item in response['items']])
            
            next_page_token = response.get('nextPageToken')

            ix += 50
            
            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status in [403, 429]:  # Quota exceeded
                current_key_index += 1
                if current_key_index >= len(api_keys):
                    print("All API keys exhausted.")
                    break
                if verbose:
                    print(f"Switching to next API key. Current key index: {current_key_index}")
            else:
                raise  # Re-raise the exception if it's not a quota error
    return channel_ids
    
def get_playlist_video_ids(playlist_id, api_keys, verbose=False):
    video_ids = []
    next_page_token = None
    current_key_index = 0

    ix = 0

    while True:
        
        sys.stdout.write(f"\r{ix}")
        sys.stdout.flush()
        
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])
            
            playlist_items = youtube.playlistItems().list(
                part='contentDetails',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            ).execute()
            
            video_ids.append([item['contentDetails']['videoId'] for item in playlist_items['items']])
            
            next_page_token = playlist_items.get('nextPageToken')

            ix += 50

            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status in [403, 404, 429]:  # Quota exceeded
                current_key_index += 1
                if current_key_index >= len(api_keys):
                    print("All API keys exhausted.")
                    break
                if verbose:
                    print(f"Switching to next API key. Current key index: {current_key_index}")
            else:
                raise  # Re-raise the exception if it's not a quota error

    return video_ids

def get_video_response(video_id, api_keys):
    for api_key in api_keys:
        try:
            youtube = build('youtube', 'v3', developerKey=api_key)
            response = youtube.videos().list(
                part='snippet,contentDetails,statistics,topicDetails',
                id=video_id
            ).execute()
            return response
        
        except HttpError as e:
            if e.resp.status == 403:
                continue
            elif e.resp.status in [400, 404, 503]:
                return
            else:
                raise e
    raise Exception("All API keys have exceeded their quota.")

In [ ]:
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build

from tqdm.notebook import tqdm as tqdm

import pandas as pd

import sys

from sqlalchemy import create_engine

In [ ]:
categories_map = {
    1: 'Film & Animation',
    2: 'Autos & Vehicles',
    10: 'Music',
    15: 'Pets & Animals',
    17: 'Sports',
    18: 'Short Movies',
    19: 'Travel & Events',
    20: 'Gaming',
    21: 'Videoblogging',
    22: 'People & Blogs',
    23: 'Comedy',
    24: 'Entertainment',
    25: 'News & Politics',
    26: 'Howto & Style',
    27: 'Education',
    28: 'Science & Technology',
    29: 'Nonprofits & Activism',
    30: 'Movies',
    31: 'Anime/Animation',
    32: 'Action/Adventure',
    33: 'Classics',
    34: 'Comedy',
    35: 'Documentary',
    36: 'Drama',
    37: 'Family',
    38: 'Foreign',
    39: 'Horror',
    40: 'Sci-Fi/Fantasy',
    41: 'Thriller',
    42: 'Shorts',
    43: 'Shows',
    44: 'Trailers'
}

In [ ]:
# Get search data
from datetime import datetime
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from sqlalchemy import create_engine
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

# end date is today
end_date = datetime.now().strftime("%Y-%m-%d")
# start date is today minus 5 years
start_date = (datetime.now() - pd.DateOffset(years=5)).strftime("%Y-%m-%d")
# date list
dates = pd.date_range(start_date, end_date, freq='D')

# Batch size for date ranges
batch_size_days = 30

# Search term
query_string = 'Need for Speed'

# Read existing data
engine = create_engine('sqlite:///../data/data.db', echo=False)

try:
    existing_videos_df = pd.read_sql('SELECT * FROM search_results', engine)
    fresh_run = False
except:
    existing_videos_df = pd.DataFrame()
    fresh_run = True

original_count = existing_videos_df.shape[0]
    
# Get API keys
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

def get_videos_for_dates(start_date, end_date, query_string, API_KEYS, existing_videos_df, fresh_run):
    video_frames = []
    current_key_index = 0
    next_page_token = None
    while True:
        try:
            youtube = build('youtube', 'v3', developerKey=API_KEYS[current_key_index])
            
            response = youtube.search().list(
                part='snippet',
                maxResults=50,
                pageToken=next_page_token,
                publishedAfter=start_date,
                publishedBefore=end_date,
                q=query_string,
                relevanceLanguage='en',
                type='video',
                videoCaption='closedCaption', # 'any', 'closedCaption', 'none'
                videoDefinition='high',
                videoDuration='long', # 'any', 'long', 'medium', 'short'
            ).execute()
            
            video_id_lists = [item.get('id').get('videoId') 
                        for item in response.get('items') 
                        if item.get('id').get('videoId')]

            video_id_strings = ','.join(video_id_lists)
            video_response = youtube.videos().list(
                part='snippet,contentDetails,statistics,topicDetails',
                id=video_id_strings
            ).execute()
            video_response_df = pd.json_normalize(video_response['items'])

            cols = [
                'id', 'snippet.publishedAt', 'snippet.channelId', 'snippet.title', 'snippet.description', 'snippet.channelTitle', 
                'snippet.tags', 'snippet.categoryId', 'snippet.liveBroadcastContent', 'snippet.localized.title', 'snippet.localized.description', 
                'snippet.defaultAudioLanguage', 'contentDetails.duration', 'contentDetails.dimension', 'contentDetails.definition', 
                'contentDetails.caption', 'statistics.viewCount', 'statistics.likeCount', 'statistics.favoriteCount', 'statistics.commentCount', 
                'topicDetails.topicCategories', 'snippet.defaultLanguage'
            ]
            
            for col in cols:
                if col not in video_response_df.columns:
                    video_response_df[col] = None
            
            video_response_df = video_response_df[cols]
            video_response_df = video_response_df.map(lambda x: x if not isinstance(x, list) else ','.join(x))
            video_response_df['queryTimestamp'] = pd.Timestamp.now()
            video_response_df['query'] = query_string
            
            if not fresh_run:
                condition = video_response_df['id'].isin(existing_videos_df['id']) & video_response_df['query'].isin(existing_videos_df['query'])
                video_response_df = video_response_df[~condition]
            
            video_frames.append(video_response_df)
            
            next_page_token = response.get('nextPageToken')        
            if not next_page_token:
                break
            
        except HttpError as e:
            if e.resp.status in [403, 429]:  # Quota exceeded
                current_key_index += 1
                if current_key_index >= len(API_KEYS):
                    print("All API keys exhausted.")
                    break
            else:
                raise  # Re-raise the exception if it's not a quota error
        except Exception as e:
            print(f"An error occurred: {e}")
            break
    
    if video_frames:
        return pd.concat(video_frames, ignore_index=True)
    return pd.DataFrame()

# Create date ranges
date_ranges = [(dates[i].strftime('%Y-%m-%dT00:00:00Z'), dates[min(i + batch_size_days, len(dates)-1)].strftime('%Y-%m-%dT00:00:00Z')) 
               for i in range(0, len(dates), batch_size_days)]

# Use ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_date_range = {executor.submit(get_videos_for_dates, start, end, query_string, API_KEYS, existing_videos_df, fresh_run): (start, end) 
                            for start, end in date_ranges}
    ix = 0
    video_frames = []
    
    pbar = tqdm(total=len(date_ranges), desc='Date Ranges')
    for future in future_to_date_range:
        try:
            video_frames.append(future.result())
            ix += video_frames[-1].shape[0]
        except Exception as exc:
            print(f"Generated an exception: {exc}")
        pbar.update(1)
        pbar.set_postfix(total=ix)
        
# Concatenate all video details into a single dataframe
if video_frames:
    videos_df = pd.concat(video_frames, ignore_index=True)
    existing_videos_df = pd.read_sql('SELECT * FROM search_results', engine)
    # combine
    videos_df = pd.concat([existing_videos_df, videos_df], ignore_index=True)
    # drop dupes
    videos_df = videos_df.drop_duplicates(subset=['id', 'query'], keep='last')
    # Convert queryTimestamp and publishedAt to str
    videos_df['queryTimestamp'] = videos_df['queryTimestamp'].astype(str)
    videos_df['snippet.publishedAt'] = videos_df['snippet.publishedAt'].astype(str)    
    # Save to database
    videos_df.to_sql('search_results', con=engine, if_exists='replace', index=False)
    # Read back from database
    videos_df = pd.read_sql('SELECT * FROM search_results', engine)
    print(videos_df)
    print(f"New videos: {videos_df.shape[0] - original_count}")
else:
    print("No data retrieved.")

In [ ]:
# Get channel data
from sqlalchemy import create_engine
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.notebook import tqdm

pd.set_option('display.width', 512)

def get_channel_statistics(channel_id, api_keys):
    for api_key in api_keys:
        try:
            youtube = build('youtube', 'v3', developerKey=api_key)
            request = youtube.channels().list(
                part='contentDetails,id,snippet,statistics,topicDetails',
                id=channel_id
            )
            response = request.execute()
            if 'items' in response and response['items']:
                items_data = pd.json_normalize(response['items'])
                items_data['queryTimestamp'] = pd.Timestamp.now()
                return items_data
            else:
                return None
        
        except HttpError as e:
            if e.resp.status == 403:
                API_KEYS.remove(api_key)
            elif e.resp.status in [400, 404, 503]:
                return
            else:
                raise e
    # raise Exception("All API keys have exceeded their quota.")

# Get API keys
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

conn = create_engine("sqlite:///../data/data.db")
search_results = pd.read_sql_query("SELECT * FROM search_results", conn)
channel_ids_from_search = search_results['snippet.channelId'].unique()
try:
    channel_statistics_table = pd.read_sql_table('channel_statistics', conn)
except ValueError:
    channel_statistics_table = pd.DataFrame(columns=['id','etag'])
channel_ids_in_table = channel_statistics_table['id'].unique()
channel_ids = list(set(channel_ids_from_search) - set(channel_ids_in_table))
# split into chunks of 50
batch_size = 50
channel_ids = [channel_ids[i:i + batch_size] for i in range(0, len(channel_ids), batch_size)]
channel_id_strings = [','.join(channel_id) for channel_id in channel_ids]
channel_statistics = [get_channel_statistics(channel_id_string, API_KEYS) 
                      for channel_id_string in tqdm(channel_id_strings)]
channels_df = pd.concat(channel_statistics)
channels_df = channels_df[~channels_df['etag'].isin(channel_statistics_table['etag'])]

for col in channels_df.columns:
    # if type col is a list, convert to string
    col_types = channels_df[col].apply(lambda x: type(x)).unique()
    if list in col_types:
        channels_df[col] = channels_df[col].apply(lambda x: ','.join(x) if type(x) == list else '')

channels_df.to_sql('channel_statistics', conn, if_exists='append', index=False)

# read back the table
channel_statistics_table = pd.read_sql_table('channel_statistics', conn)
channel_statistics_table

In [ ]:
# get playlist data
from sqlalchemy import create_engine
import pandas as pd
from tqdm.notebook import tqdm
import sys
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

pd.set_option('display.width', 1024)
pd.set_option('display.max_columns', 32)

# Get API keys
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

# Function to create a YouTube API client with a given API key
def get_youtube_client(api_key):
    return build('youtube', 'v3', developerKey=api_key)

# Function to get the next API key, cycling through the list
def get_next_api_key(current_key_index, api_keys):
    return (current_key_index + 1) % len(api_keys)

def get_playlist_video_ids(playlist_id, api_keys, current_key_index, verbose=False):
    playlist_items_dfs = []
    next_page_token = None

    ix = 0

    while True:
        sys.stdout.write(f"\r{ix}          ")
        sys.stdout.flush()

        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])

            playlist_items = youtube.playlistItems().list(
                part='contentDetails,snippet',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            ).execute()

            items_df = pd.json_normalize(playlist_items['items'])
            playlist_items_dfs.append(items_df)

            next_page_token = playlist_items.get('nextPageToken')

            ix += len(items_df)

            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                current_key_index = get_next_api_key(current_key_index, api_keys)
                if current_key_index == 0:
                    print("All API keys exhausted.")
                    break
            elif e.resp.status == 404:
                print("Playlist not found.")
                break
            else:
                raise  # Re-raise the exception if it's not a quota error
    if playlist_items_dfs:
        playlist_items_df = pd.concat(playlist_items_dfs, ignore_index=True)
        return playlist_items_df, current_key_index
    return pd.DataFrame(), current_key_index
    
# Read existing data
engine = create_engine('sqlite:///../data/data.db', echo=False)
channel_statistics_df = pd.read_sql_table('channel_statistics', engine)
uploads_playlist_ids_from_channels = channel_statistics_df['contentDetails.relatedPlaylists.uploads'].unique()

try:
    channel_uploads_df = pd.read_sql_table('channel_uploads', engine)
except ValueError:
    channel_uploads_df = pd.DataFrame(columns=['snippet.playlistId'])
    
channel_uploads_playlist_ids = channel_uploads_df['snippet.playlistId'].unique()
playlist_ids = set(uploads_playlist_ids_from_channels) - set(channel_uploads_playlist_ids)

# sum channel_statistics_df['statistics.videoCount'] where contentDetails.relatedPlaylists.uploads is in playlist_ids
unwritten_channel_statistics = channel_statistics_df[channel_statistics_df['contentDetails.relatedPlaylists.uploads'].isin(playlist_ids)]
video_count = unwritten_channel_statistics['statistics.videoCount'].astype(int).sum()
pbar = tqdm(total=video_count, desc='Playlist Videos', unit='videos')
current_key_index = 0
for playlist_id in playlist_ids:
    message = f'Key Usage: {current_key_index} / {len(API_KEYS)}'
    pbar.set_postfix_str(message)
    playlist_video_ids_df, current_key_index = get_playlist_video_ids(playlist_id, API_KEYS, current_key_index)
    playlist_video_ids_df.to_sql('channel_uploads', con=engine, if_exists='append', index=False)
    pbar.update(len(playlist_video_ids_df))

# read back table
playlist_df = pd.read_sql_table('channel_uploads', engine)
playlist_df

In [ ]:
# get video data
from sqlalchemy import create_engine
import pandas as pd
from tqdm.notebook import tqdm

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 512)
pd.set_option('display.max_colwidth', 32)

from IPython.display import HTML

def multi_display(*dfs):
    for df in dfs:
        display(HTML(df.to_html()))

# Get API keys
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

# Read existing data
engine = create_engine('sqlite:///../data/data.db', echo=False)
channel_uploads_df = pd.read_sql_table('channel_uploads', engine)
channel_uploads_video_ids = channel_uploads_df['contentDetails.videoId'].tolist()

try:
    videos_df = pd.read_sql_table('videos', engine)
except ValueError:
    videos_df = pd.DataFrame(columns=['id'])
# Convert the list of ids to a set
videos_df_ids_set = set(videos_df['id'].tolist())
# Use the set for the membership check
video_ids = [video_id for video_id in tqdm(channel_uploads_video_ids) if video_id not in videos_df_ids_set]
# break into chunks of 50
video_ids_chunks = [video_ids[i:i + 50] for i in range(0, len(video_ids), 50)]
video_ids_strings = [','.join(chunk) for chunk in video_ids_chunks]

additional_columns = ['kind', 'etag', 'id', 'snippet.publishedAt', 'snippet.channelId', 'snippet.title', 'snippet.description', 'snippet.thumbnails.default.url',
                      'snippet.thumbnails.default.width', 'snippet.thumbnails.default.height', 'snippet.thumbnails.medium.url', 'snippet.thumbnails.medium.width', 
                      'snippet.thumbnails.medium.height', 'snippet.thumbnails.high.url', 'snippet.thumbnails.high.width', 'snippet.thumbnails.high.height', 
                      'snippet.thumbnails.standard.url', 'snippet.thumbnails.standard.width', 'snippet.thumbnails.standard.height', 'snippet.channelTitle', 'snippet.tags', 
                      'snippet.categoryId', 'snippet.liveBroadcastContent', 'snippet.localized.title', 'snippet.localized.description', 'snippet.defaultAudioLanguage', 
                      'contentDetails.duration', 'contentDetails.dimension', 'contentDetails.definition', 'contentDetails.caption', 'contentDetails.licensedContent', 
                      'contentDetails.projection', 'statistics.viewCount', 'statistics.likeCount', 'statistics.favoriteCount', 'statistics.commentCount', 
                      'topicDetails.topicCategories', 'snippet.thumbnails.maxres.url', 'snippet.thumbnails.maxres.width', 'snippet.thumbnails.maxres.height', 
                      'snippet.defaultLanguage', 'liveStreamingDetails.actualStartTime', 'liveStreamingDetails.actualEndTime', 'liveStreamingDetails.activeLiveChatId',
                      'liveStreamingDetails.scheduledStartTime', 'contentDetails.contentRating.ytRating','contentDetails.regionRestriction.blocked',
                      'liveStreamingDetails.concurrentViewers', 'liveStreamingDetails.scheduledEndTime', 'liveStreamingDetails.scheduledStartTime',
                      'contentDetails.regionRestriction.allowed']

key_index = 0
pbar = tqdm(total=len(video_ids_strings), desc='Video Data')
video_added_count = 0
for video_ids_string in list(video_ids_strings):
    while True:
        try:
            youtube = build('youtube', 'v3', developerKey=API_KEYS[key_index])
            response = youtube.videos().list(
                part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
                id=video_ids_string
            ).execute()
            videos_response_df = pd.json_normalize(response['items'])
            for col in videos_response_df.columns:
                # if type col is a list, convert to string
                col_types = videos_response_df[col].apply(lambda x: type(x)).unique()
                if list in col_types:
                    videos_response_df[col] = videos_response_df[col].apply(lambda x: ','.join(x) if type(x) == list else '')
            for col in additional_columns:
                if col not in videos_response_df.columns:
                    videos_response_df[col] = None
            videos_reponse_df = pd.concat([videos_df, videos_response_df], ignore_index=True)
            pbar.update(1)
            video_added_count += len(videos_response_df)
            pbar.set_postfix_str(f"Videos Added: {video_added_count}")
            videos_response_df.to_sql('videos', engine, if_exists='append', index=False)
            break
        except HttpError as e:
            if e.resp.status == 403:
                key_index += 1
                if key_index == len(API_KEYS):
                    raise 

In [ ]:
# read videos
engine = create_engine('sqlite:///../data/data.db', echo=False)

videos_df = pd.read_sql_table('videos', engine)
videos_df

In [ ]:
title_df = videos_df.copy()
# default languge contains en
title_df = title_df[title_df['snippet.defaultLanguage'].str.contains('en', na=False)]
title_df = title_df[title_df['snippet.defaultAudioLanguage'].str.contains('en', na=False)]
title_df

In [ ]:
search_results_df = pd.read_sql_table('search_results', engine)
search_results_df

In [ ]:
search_results_df['query'].value_counts()